<a href="https://colab.research.google.com/github/GabrielPSMartins/assistente_pronuncia/blob/main/assistente_de_voz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Gravação de áudio com Python e um pouco de JavaScript

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def gravar_pronuncia(sec=5):
    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    audio = b64decode(js_result.split(',')[1])
    file_name = 'user_voice.wav'
    with open(file_name, 'wb') as f:
      f.write(audio)
    return f'/content/{file_name}'

print('Gravando...\n')
record_file = gravar_pronuncia(sec=5)
display(Audio(record_file, autoplay=True))

Gravando...



<IPython.core.display.Javascript object>

In [ ]:
## Reconhecimento de Fala com Whisper

!pip install git+https://github.com/openai/whisper.git -q

In [ ]:
import whisper

print("Carregando o modelo Whisper...")
model = whisper.load_model("small")

result = model.transcribe(record_file, fp16=False, language="en")
texto_transcrito = result["text"]

print("\nO que o Whisper entendeu:")
print(f"-> {texto_transcrito}")


Carregando o modelo Whisper...


100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 83.9MiB/s]



O que o Whisper entendeu:
->  test sound test Let's go


In [ ]:
# Sintetizando As respostas como voz (gtts)

!pip install gTTS

In [ ]:
from gtts import gTTS

tts = gTTS(text=texto_transcrito, lang="en", slow=False)
response_audio = "/content/native_feedback.wav"

tts.save(response_audio)

print("\nOuça a pronúncia nativa:")
display(Audio(response_audio, autoplay=True))


Ouça a pronúncia nativa:
